**Mount Drive in Colab and install dependencies**

In [ ]:
# 1. Mount Google Drive 
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repo
import os 
repo_path = '/content/darts-cards'
if not os.path.exists(repo_path):
    !git clone https://github.com/javiergarciaduran/darts-cards.git {repo_path}
else: 
    !git -C {repo_path} pull
%cd {repo_path}

In [ ]:
# 3. Install extra dependencies 
!pip install -q graphviz wandb
!apt-get install -q graphviz

In [ ]:
# 4. Symlink data from Drive into repo
!ln -sfn /content/drive/MyDrive/cards ./data/cards

# 5. Verify that the dataset is correctly found
!python -c "
from datasets.cards import get_cards
tr, nc = get_cards('./data/cards', split='train')
va, _  = get_cards('./data/cards', split='val')
print(f'train={len(tr)} val={len(va)} classes={nc}')
"

**Perform the search over the different architectures (might take several hours to complete)**

In [ ]:
# 0. Create output directories on Drive
!mkdir -p /content/drive/MyDrive/darts_experiments/searchs
!mkdir -p /content/drive/MyDrive/darts_experiments/augments
!mkdir -p /content/drive/MyDrive/darts_logs

In [ ]:
# 1. Run search
!python search.py \
    --name cards_search_v1 \
    --dataset cards \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/searchs \
    --batch_size 64 \
    --init_channels 16 \
    --layers 8 \
    --epochs 50 \
    --w_lr 0.025 \
    --w_lr_min 0.001 \
    --alpha_lr 3e-4 \
    --alpha_weight_decay 1e-3 \
    --print_freq 50 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/search_v1.log

**⚠️ Manual checkpoint — DO NOT SKIP**

Run the cell below to inspect the genotype. Then:

1. Review the printed genotype — does it have a variety of operations?
2. Open the architecture plots in Drive at `darts_experiments/searchs/cards_search_v1/plots/`
3. If the genotype looks reasonable, paste it into `genotypes.py` as `CARDS_V1`
4. Commit and push:
   ```
   git add genotypes.py
   git commit -m "add CARDS_V1 genotype from search run v1 (seed=42, 50 epochs)"
   git push
   ```
5. Pull the update in Colab: `!git pull`
6. Then run the augment cell

In [ ]:
# 2. Inspect genotype and check for skip-connection collapse
genotype_path = '/content/drive/MyDrive/darts_experiments/searchs/cards_search_v1/genotype.txt'
with open(genotype_path) as f:
    genotype_str = f.read()
print(genotype_str)

print('\n--- Checking for skip-connect dominance ---')
skip_count = genotype_str.count('skip_connect')
print(f'skip_connect appears {skip_count} times in the genotype')
if skip_count > 4:
    print('WARNING: high skip-connect count — search may have collapsed')
else:
    print('OK: skip-connect count looks reasonable')

In [ ]:
# 3. Run augment (only after completing the manual checkpoint above)
!python augment.py \
    --name cards_augment_v1 \
    --dataset cards \
    --data_path ./data/cards \
    --path /content/drive/MyDrive/darts_experiments/augments \
    --batch_size 96 \
    --init_channels 24 \
    --layers 14 \
    --epochs 200 \
    --lr 0.025 \
    --aux_weight 0.4 \
    --drop_path_prob 0.2 \
    --cutout_length 8 \
    --genotype CARDS_V1 \
    --print_freq 50 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/augment_v1.log